<a href="https://colab.research.google.com/github/zyrquen-engine/automatic-memory/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import os
import sys

# 1. เข้าไปในโฟลเดอร์โปรเจกต์
os.chdir('zyntheor_continuation')
sys.path.append('.')

try:
    # 2. โหลดคลาสโมเดลและ Tokenizer ให้ตรงกับชื่อจริงในโค้ดของคุณ
    from qplus_native.model import ZyntheorTransformer, ModelConfig
    from qplus_native.tokenizer import SovereignTokenizer
    print("✅ โหลดโค้ดสถาปัตยกรรมและ Tokenizer สำเร็จ!")

    # 3. ตั้งค่า Config ตามสเปก 41.5M
    config = ModelConfig(vocab_size=16384, d_model=512, n_layers=10, n_heads=8, d_ffn=1472)
    model = ZyntheorTransformer(config)

    # 4. โหลด Weight 159M เข้าไป
    checkpoint_path = 'native_data/model_checkpoint_target_41m.pt'
    print(f"⏳ กำลังโหลด Weight จาก {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print("✅ โหลด Weight (model_state_dict) สำเร็จ!")
    else:
        model.load_state_dict(checkpoint)
        print("✅ โหลด Weight (direct) สำเร็จ!")

    model.eval()

    # 5. แสดงจำนวนพารามิเตอร์ที่โหลดได้จริง
    total_params = sum(p.numel() for p in model.parameters())
    print(f"🎯 จำนวนพารามิเตอร์ที่โหลดเข้า Memory จริง: {total_params:,} ตัว")

    # 6. ลอง Gen คำตอบ!
    print("\n--- 🧠 ลองให้ Native AI 41.5M พิมพ์คำตอบ (Zero-Fake Test) ---")

    tokenizer = SovereignTokenizer(vocab_size=16384)
    prompt = "ZYNTHEOR is"
    print(f"Prompt: '{prompt}'")

    # แปลงข้อความเป็นตัวเลข
    input_ids = tokenizer.encode(prompt)
    input_tensor = torch.tensor([input_ids], dtype=torch.long)

    print("⏳ กำลังประมวลผล (Forward Pass)...")
    with torch.no_grad():
        # ตรวจสอบว่ามีฟังก์ชัน generate ในโมเดลหรือไม่
        if hasattr(model, 'generate'):
            output = model.generate(input_tensor, max_new_tokens=15)
            generated_text = tokenizer.decode(output[0].tolist())
            print(f"\n✅ คำตอบจากโมเดล: {generated_text}")
        else:
            # ถ้าไม่มีฟังก์ชัน generate ให้ทำแบบ Manual Loop
            generated_ids = input_ids.copy()
            for _ in range(10): # Gen เพิ่ม 10 token
                logits, _ = model(torch.tensor([generated_ids], dtype=torch.long))
                next_token_logits = logits[0, -1, :]
                next_token_id = torch.argmax(next_token_logits, dim=-1).item()
                generated_ids.append(next_token_id)

            generated_text = tokenizer.decode(generated_ids)
            print(f"\n✅ คำตอบจากโมเดล (Manual Gen): {generated_text}")

except Exception as e:
    print(f"❌ เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

FileNotFoundError: [Errno 2] No such file or directory: 'zyntheor_continuation'

In [ ]:

%cd /content/zyntheor_continuation

!echo "=== PROJECT SIZE ==="
!du -sh .

!echo "=== PROJECT FILES ==="
!find . -maxdepth 3 -type f -not -path '*/.git/*' -printf '%s %p\n' | sort -k2

!echo "=== LARGE FILES ==="
!find . -type f -size +20M -not -path '*/.git/*' -printf '%s %p\n'

!echo "=== GIT STATUS ==="
!git status 2>/dev/null || echo "No Git repository yet"

In [106]:
!cd /content/zyntheor_continuation && du -sh . && find . -maxdepth 2 -type f | head -40

645M	.
./.npmrc
./assets/banner-bottom.gif
./assets/banner-top.gif
./tsconfig.json
./.env.example
./CONTRIBUTING.md
./package.json
./REVIEW-AND-IMPROVEMENT-REPORT.md
./workspace/solution.js
./backups/zyntheor-data-202611111610.tgz
./IMPLEMENTATION_REPORT.md
./native_data/security_rbac_integrity_report.json
./native_data/memory_store.json
./native_data/identity_trust_test_report.json
./native_data/knowledge_intelligence_test_report.json
./native_data/revoked_jti.jsonl
./native_data/knowledge_e2e_report.json
./native_data/chat_sessions.json
./native_data/audit_log.jsonl
./native_data/knowledge_live_http_report.json
./native_data/revoked_jti_index.json
./native_data/model_checkpoint.pre-parameter-audit-20261112.pt
./native_data/model_checkpoint_target_41m.pt.backup-1794497240
./native_data/model_checkpoint_target_41m.pt
./native_data/model_checkpoint_target_41m.pt.backup-1794495866
./native_data/model_checkpoint_v2.pt
./native_data/model_checkpoint.pt
./native_data/model_checkpoint_target

In [ ]:
# ค้นหาชื่อคลาสทั้งหมดที่อยู่ในไฟล์ model.py และ lm_v2.py
!echo "--- คลาสใน model.py ---"
!grep "^class " qplus_native/model.py

!echo "\n--- คลาสใน lm_v2.py ---"
!grep "^class " qplus_native/lm_v2.py

!echo "\n--- ฟังก์ชันใน model.py ---"
!grep "^def " qplus_native/model.py

--- คลาสใน model.py ---
class ModelConfig:
class NativeNeuralModel:
\n--- คลาสใน lm_v2.py ---
class LMConfig:
class NativeLM:
class AdamW:
\n--- ฟังก์ชันใน model.py ---


In [ ]:
# ตรวจสอบคลาสที่มีอยู่ในไฟล์ tokenizer.py
with open('qplus_native/tokenizer.py', 'r') as f:
    for line in f:
        if 'class ' in line or 'def ' in line:
            print(line.strip())

class SovereignTokenizer:
def __init__(self, vocab_size: int = 16384):
def _build_vocab(self):
def encode(self, text: str) -> List[int]:
def decode(self, token_ids: List[int]) -> str:


In [ ]:
import torch
import os
import sys
import inspect

# 1. ตรวจสอบตำแหน่งโฟลเดอร์
%cd /content
os.chdir('zyntheor_continuation')
sys.path.append('.')
print(f"📂 ตำแหน่งปัจจุบัน: {os.getcwd()}")

try:
    # 2. โหลดคลาส
    from qplus_native.model import NativeNeuralModel, ModelConfig
    from qplus_native.tokenizer import SovereignTokenizer
    print("✅ โหลดโค้ดสถาปัตยกรรมและ Tokenizer สำเร็จ!")

    # 3. ส่องดูว่าคลาส ModelConfig รับพารามิเตอร์อะไรบ้าง
    sig = inspect.signature(ModelConfig.__init__)
    print(f"🔍 พารามิเตอร์ที่ ModelConfig ต้องการ: {list(sig.parameters.keys())}")

    # 4. สร้าง Config โดยใช้ชื่อ 'vocab' แทน 'vocab_size'
    # (ถ้าในโค้ดใช้ชื่ออื่น เราจะได้เห็นจาก print ด้านบน)
    config = ModelConfig(
        vocab=16384,       # เปลี่ยนจาก vocab_size เป็น vocab
        d_model=512,
        n_layers=10,
        n_heads=8,
        d_ffn=1472,
        ctx=64             # เพิ่ม ctx เผื่อไว้ (ตามข้อมูลใน JSON)
    )
    model = NativeNeuralModel(config)

    # 5. โหลด Weight 159M เข้าไป
    checkpoint_path = 'native_data/model_checkpoint_target_41m.pt'
    print(f"⏳ กำลังโหลด Weight จาก {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # ตรวจสอบโครงสร้าง Checkpoint
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print("✅ โหลด Weight (model_state_dict) สำเร็จ!")
    elif 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
        print("✅ โหลด Weight (state_dict) สำเร็จ!")
    else:
        model.load_state_dict(checkpoint)
        print("✅ โหลด Weight (direct) สำเร็จ!")

    model.eval()

    # 6. แสดงจำนวนพารามิเตอร์ที่โหลดได้จริง
    total_params = sum(p.numel() for p in model.parameters())
    print(f"🎯 จำนวนพารามิเตอร์ที่โหลดเข้า Memory จริง: {total_params:,} ตัว")

    # 7. ลอง Gen คำตอบ!
    print("\n--- 🧠 ลองให้ Native AI 41.5M พิมพ์คำตอบ (Zero-Fake Test) ---")
    tokenizer = SovereignTokenizer(vocab_size=16384)
    prompt = "ZYNTHEOR is"
    print(f"Prompt: '{prompt}'")

    input_ids = tokenizer.encode(prompt)

    # แปลงให้เป็น Tensor 2D (batch_size=1, seq_len)
    input_tensor = torch.tensor([input_ids], dtype=torch.long)

    print("⏳ กำลังประมวลผล (Forward Pass)...")
    with torch.no_grad():
        generated_ids = input_ids.copy()
        for _ in range(10):
            # ส่ง Tensor เข้าโมเดล
            logits, _ = model(torch.tensor([generated_ids], dtype=torch.long))
            next_token_logits = logits[0, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()
            generated_ids.append(next_token_id)

        generated_text = tokenizer.decode(generated_ids)
        print(f"\n✅ คำตอบจากโมเดล: {generated_text}")

except Exception as e:
    print(f"❌ เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

/content
📂 ตำแหน่งปัจจุบัน: /content/zyntheor_continuation
✅ โหลดโค้ดสถาปัตยกรรมและ Tokenizer สำเร็จ!
🔍 พารามิเตอร์ที่ ModelConfig ต้องการ: ['self']
❌ เกิดข้อผิดพลาด: ModelConfig.__init__() got an unexpected keyword argument 'vocab'


Traceback (most recent call last):
  File "/tmp/ipykernel_10080/2532591644.py", line 24, in <cell line: 0>
    config = ModelConfig(
        vocab=16384,       # เปลี่ยนจาก vocab_size เป็น vocab
    ...<4 lines>...
        ctx=64             # เพิ่ม ctx เผื่อไว้ (ตามข้อมูลใน JSON)
    )
TypeError: ModelConfig.__init__() got an unexpected keyword argument 'vocab'


In [ ]:
import torch
import os
import sys

# 1. ตรวจสอบตำแหน่งโฟลเดอร์
%cd /content
os.chdir('zyntheor_continuation')
sys.path.append('.')
print(f"📂 ตำแหน่งปัจจุบัน: {os.getcwd()}")

try:
    # 2. โหลดคลาส
    from qplus_native.model import NativeNeuralModel, ModelConfig
    from qplus_native.tokenizer import SovereignTokenizer
    print("✅ โหลดโค้ดสถาปัตยกรรมและ Tokenizer สำเร็จ!")

    # 3. สร้าง Config เปล่าๆ ขึ้นมาก่อน
    config = ModelConfig()

    # 4. แอบแก้ค่า Config ข้างในให้เป็นสเปก 41.5M (ถ้าตัวแปรมีอยู่จริง)
    target_config = {
        'vocab': 16384, 'vocab_size': 16384,
        'd_model': 512,
        'n_layers': 10,
        'n_heads': 8,
        'd_ffn': 1472,
        'ctx': 64
    }
    print("🔧 กำลังปรับแต่ง Config ให้เป็น 41.5M...")
    for key, val in target_config.items():
        if hasattr(config, key):
            setattr(config, key, val)
            print(f"   - ตั้งค่า {key} = {val}")

    # 5. สร้างโมเดล
    model = NativeNeuralModel(config)

    # 6. โหลด Weight 159M
    checkpoint_path = 'native_data/model_checkpoint_target_41m.pt'
    print(f"\n⏳ กำลังโหลด Weight จาก {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    elif 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print("✅ โหลด Weight สำเร็จ!")

    model.eval()

    # 7. แสดงจำนวนพารามิเตอร์ที่โหลดได้จริง
    total_params = sum(p.numel() for p in model.parameters())
    print(f"🎯 จำนวนพารามิเตอร์ที่โหลดเข้า Memory จริง: {total_params:,} ตัว")

    # 8. ลอง Gen คำตอบ!
    print("\n--- 🧠 ลองให้ Native AI 41.5M พิมพ์คำตอบ (Zero-Fake Test) ---")
    tokenizer = SovereignTokenizer(vocab_size=16384)
    prompt = "ZYNTHEOR is"
    print(f"Prompt: '{prompt}'")

    input_ids = tokenizer.encode(prompt)
    input_tensor = torch.tensor([input_ids], dtype=torch.long)

    print("⏳ กำลังประมวลผล (Forward Pass)...")
    with torch.no_grad():
        generated_ids = input_ids.copy()
        for _ in range(10):
            logits, _ = model(torch.tensor([generated_ids], dtype=torch.long))
            next_token_logits = logits[0, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()
            generated_ids.append(next_token_id)

        generated_text = tokenizer.decode(generated_ids)
        print(f"\n✅ คำตอบจากโมเดล: {generated_text}")

except Exception as e:
    print(f"\n❌ เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
import torch
import os
import sys

# 1. ตรวจสอบตำแหน่งโฟลเดอร์
%cd /content
os.chdir('zyntheor_continuation')
sys.path.append('.')
print(f"📂 ตำแหน่งปัจจุบัน: {os.getcwd()}")

try:
    # 2. โหลดคลาส
    from qplus_native.model import NativeNeuralModel
    from qplus_native.tokenizer import SovereignTokenizer
    print("✅ โหลดโค้ดสถาปัตยกรรมสำเร็จ!")

    checkpoint_path = 'native_data/model_checkpoint_target_41m.pt'
    print(f"⏳ กำลังโหลดโมเดลจาก: {checkpoint_path}...")

    # 3. สร้างโมเดลโดยส่ง Path ของไฟล์ (String) เข้าไปตรงๆ!
    model = NativeNeuralModel(checkpoint_path)
    model.eval()
    print("✅ โหลด Weight เข้า Memory สำเร็จ!")

    # 4. แสดงจำนวนพารามิเตอร์
    total_params = sum(p.numel() for p in model.parameters())
    print(f"🎯 จำนวนพารามิเตอร์ที่โหลดได้จริง: {total_params:,} ตัว")

    # 5. ลอง Gen คำตอบ!
    print("\n--- 🧠 ลองให้ Native AI 41.5M พิมพ์คำตอบ (Zero-Fake Test) ---")
    tokenizer = SovereignTokenizer(vocab_size=16384)
    prompt = "ZYNTHEOR is"
    print(f"Prompt: '{prompt}'")

    input_ids = tokenizer.encode(prompt)

    print("⏳ กำลังประมวลผล (Forward Pass)...")
    with torch.no_grad():
        generated_ids = input_ids.copy()
        for _ in range(10): # Gen 10 token
            # ส่ง Tensor เข้าโมเดล
            logits, _ = model(torch.tensor([generated_ids], dtype=torch.long))
            next_token_logits = logits[0, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()
            generated_ids.append(next_token_id)

        generated_text = tokenizer.decode(generated_ids)
        print(f"\n✅ คำตอบจากโมเดล: {generated_text}")

except Exception as e:
    print(f"\n❌ เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

/content
📂 ตำแหน่งปัจจุบัน: /content/zyntheor_continuation
✅ โหลดโค้ดสถาปัตยกรรมสำเร็จ!
⏳ กำลังโหลดโมเดลจาก: native_data/model_checkpoint_target_41m.pt...

❌ เกิดข้อผิดพลาด: FAIL_CLOSED: Failed to initialize native neural checkpoint: FAIL_CLOSED: Active parameter mismatch. Expected 231168, metadata total=41527808, trained_subgraph=41527808


Traceback (most recent call last):
  File "/content/zyntheor_continuation/qplus_native/model.py", line 91, in load_checkpoint
    raise ValueError(
    ...<2 lines>...
    )
ValueError: FAIL_CLOSED: Active parameter mismatch. Expected 231168, metadata total=41527808, trained_subgraph=41527808

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_10080/3297965301.py", line 21, in <cell line: 0>
    model = NativeNeuralModel(checkpoint_path)
  File "/content/zyntheor_continuation/qplus_native/model.py", line 65, in __init__
    self.load_checkpoint()
    ~~~~~~~~~~~~~~~~~~~~^^
  File "/content/zyntheor_continuation/qplus_native/model.py", line 121, in load_checkpoint
    raise RuntimeError(f"FAIL_CLOSED: Failed to initialize native neural checkpoint: {str(e)}")
RuntimeError: FAIL_CLOSED: Failed to initialize native neural checkpoint: FAIL_CLOSED: Active parameter mismatch. Expected 231168, metadata total=41527808, 

In [107]:
%cd /content/zyntheor_continuation

!echo "=== TOP LEVEL DIRECTORIES ==="
!find . -maxdepth 1 -type d

!echo "=== SOURCE FILES ==="
!find . -maxdepth 2 -type f \
  \( -name "*.py" -o -name "*.js" -o -name "*.ts" -o -name "*.tsx" -o -name "*.json" \) \
  -not -path './native_data/*' \
  -not -path './backups/*' \
  | head -100

!echo "=== LARGE DIRECTORIES ==="
!du -h --max-depth=1 . | sort -hr

/content/zyntheor_continuation
=== TOP LEVEL DIRECTORIES ===
.
./assets
./workspace
./backups
./native_data
./artifacts
./.github
./public
./tests
./qplus_native
./src
./lib
=== SOURCE FILES ===
./tsconfig.json
./package.json
./workspace/solution.js
./artifacts/lm-v2-eval.json
./artifacts/lm-v2-corpus-manifest.json
./artifacts/native-parameter-integrity-results.json
./artifacts/target-training-evidence.json
./artifacts/lm-v2-training-evidence.json
./artifacts/training-evidence.json
./artifacts/continued-training-evidence.json
./vite.config.ts
./public/manus-routes.json
./tests/identity-trust.test.ts
./tests/native-lm-v2.test.py
./tests/sovereignty.test.ts
./tests/native-parameter-integrity.test.py
./tests/knowledge_e2e.test.ts
./metadata.json
./qplus_native/checkpoint.py
./qplus_native/engine.py
./qplus_native/architecture.py
./qplus_native/init_weights.py
./qplus_native/train_target.py
./qplus_native/tokenizer.py
./qplus_native/bridge.py
./qplus_native/corpus.py
./qplus_native/train_l

In [108]:

%cd /content/zyntheor_continuation

content = """
.env
.env.*
!.env.example

node_modules/
dist/
build/
.venv/
venv/
__pycache__/
*.pyc
.pytest_cache/
.cache/

native_data/
backups/

*.pt
*.pth
*.ckpt
*.safetensors
*.bin
*.zip
*.tgz

*.log
*.tmp
*.pem
*.key
credentials/
secrets/

# Review JSON evidence before publishing
artifacts/*.json
"""

with open(".gitignore", "w") as f:
    f.write(content.strip() + "\n")

print("Created .gitignore")

/content/zyntheor_continuation
Created .gitignore


In [109]:
%cd /content/zyntheor_continuation

!git init
!git add -A
!git status --short
!git diff --cached --stat

/content/zyntheor_continuation
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/zyntheor_continuation/.git/
A  .env.example
A  .github/dependabot.yml
A  .github/workflows/api-integration.yml
A  .github/workflows/ci.yml
A  .github/workflows/codeql.yml
A  .github/workflows/dependency-review.yml
A  .github/workflows/e2e.yml
A  .github/workflows/license-audit.yml
A  .github/workflows/python-ci.yml
A  .github/workflows/python-security.yml
A  .github/workflows/sbom.yml
A  .github/workflows/scorecard.yml
A  .github/workf

In [110]:
%cd /content/zyntheor_continuation

!git diff --cached --check

print("=== ENV EXAMPLE ===")
!cat .env.example

print("=== CONFIGURATION ===")
!cat metadata.json
!cat public/manus-routes.json

print("=== POSSIBLE SECRET SCAN ===")
!git diff --cached --no-ext-diff -U0 | grep -Ein \
'(ghp_[A-Za-z0-9]{20,}|github_pat_|sk-[A-Za-z0-9]{20,}|-----BEGIN .*PRIVATE KEY-----|AKIA[0-9A-Z]{16})' \
|| echo "No common secret patterns detected"

/content/zyntheor_continuation
artifacts/PARAMETER-AUDIT-AND-FULL-SOURCE-REVIEW-2026-11-12.md:3: trailing whitespace.
+Date: 2026-11-12  
index.html:21: new blank line at EOF.
lib/CommercialManager.ts:152: trailing whitespace.
+    
lib/KnowledgeWorkspace.ts:3: trailing whitespace.
+ * 
lib/KnowledgeWorkspace.ts:6: trailing whitespace.
+ * 
lib/KnowledgeWorkspace.ts:13: trailing whitespace.
+ * 
lib/NativeMemoryController.ts:5: trailing whitespace.
+ * 
lib/SovereignIdentity.ts:4: trailing whitespace.
+ * 
lib/SovereignIdentity.ts:38: trailing whitespace.
+  code: 
lib/providers.ts:116: trailing whitespace.
+        
qplus_native/bridge.py:273: trailing whitespace.
+    
qplus_native/engine.py:482: new blank line at EOF.
qplus_native/model.py:63: trailing whitespace.
+        
qplus_native/model.py:72: trailing whitespace.
+        
qplus_native/model.py:78: trailing whitespace.
+                
qplus_native/model.py:86: trailing whitespace.
+                
qplus_native/model.py:95: